In [1]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from os.path import exists

sys.path.append('../..')

In [3]:
from stable_baselines3 import PPO, DQN

In [4]:
from vimms.Common import POSITIVE, load_obj, save_obj
from vimms.Evaluation import evaluate_real
from vimms.Chemicals import ChemicalMixtureFromMZML
from vimms.Roi import RoiBuilderParams

from vimms.MassSpec import IndependentMassSpectrometer
from vimms.Controller import TopNController
from vimms.Environment import Environment

from vimms_gym.env import DDAEnv
from vimms_gym.evaluation import run_method
from vimms_gym.common import METHOD_PPO, METHOD_TOPN, METHOD_RANDOM

/opt/anaconda3/envs/vimms-gym/lib/python3.9/site-packages/statsmodels/compat/pandas.py:61: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import Int64Index as NumericIndex
/opt/anaconda3/envs/vimms-gym/lib/python3.9/site-packages/psims/mzmlb/writer.py:15: UserWarning: hdf5plugin is missing! Only the slower GZIP compression scheme will be available! Please install hdf5plugin to be able to use Blosc.
  warnings.warn(


# 1. Parameters

In [5]:
mz_range = (70, 1000)
rt_range = (0, 1440)

In [6]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]

In [7]:
isolation_window = 0.7
N = 10
rt_tol = 120
small_rt_tol = 15
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE

In [8]:
params = {
    'chemical_creator': None,
    'noise': {
        'enable_spike_noise': False,
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
    }
}

In [9]:
max_peaks = 200
in_dir = '../QCB_chems/results'

In [10]:
deterministic = True

# 2. Evaluation on QCB data

In [11]:
env_name = 'DDAEnv'

In [12]:
eval_dir = 'evaluation_QCB'
methods = [
    METHOD_PPO,
    METHOD_TOPN,
    METHOD_RANDOM,    
]
out_dir = eval_dir
in_dir, out_dir

('../QCB_chems/results', 'evaluation_QCB')

In [13]:
intensity_threshold = 0.5

#### Load pre-processed QCB chemicals

In [14]:
fullscan_file = '../fullscan_QCB.mzML'

In [15]:
# min_roi_intensity = 0
# min_roi_length = 0

# min_roi_intensity = 500
# min_roi_length = 0
# at_least_one_point_above = 5000

min_roi_intensity = 0
min_roi_length = 3
at_least_one_point_above = 1000

In [16]:
filename = '../datasets_%d_%d_%d.p' % (min_roi_intensity, min_roi_length, at_least_one_point_above)

if exists(filename):
    chemicals = load_obj(filename)
    print(len(chemicals))
else:
    rp = RoiBuilderParams(min_roi_intensity=min_roi_intensity, min_roi_length=min_roi_length, 
                   at_least_one_point_above=at_least_one_point_above)
    cm = ChemicalMixtureFromMZML(fullscan_file, roi_params=rp)
    chemicals = cm.sample(None, 2, source_polarity=ionisation_mode)
    print(len(chemicals))
    save_obj(chemicals, filename)

43107


#### Filter chemicals by mz and RT range

In [17]:
filtered = []
for chem in chemicals:
    if (min_mz < chem.isotopes[0][0] < max_mz) and (min_rt < chem.rt < max_rt):
        filtered.append(chem)
        
len(filtered)

43060

In [18]:
filtered_chem_list = [filtered]

#### Test different methods

In [19]:
copy_params = dict(params)
copy_params['env']['rt_tol'] = rt_tol

for method in methods:
    banner = 'method = %s max_peaks = %d rt_tol = %d' % (method, max_peaks, rt_tol)
    print(banner)
    print()
    
    copy_params = dict(params)
    if method == 'PPO':
        fname = os.path.join(in_dir, '%s_%s.zip' % (env_name, method))
        model = PPO.load(fname)
        copy_params['env']['rt_tol'] = rt_tol
    elif method == 'DQN':
        fname = os.path.join(in_dir, '%s_%s.zip' % (env_name, method))
        model = DQN.load(fname)
        copy_params['env']['rt_tol'] = rt_tol        
    else:
        model = None
        copy_params['env']['rt_tol'] = small_rt_tol        

    env = DDAEnv(max_peaks, copy_params)
    run_method(env_name, copy_params, max_peaks, filtered_chem_list, method, out_dir, mzml_prefix='QCB',
               N=10, min_ms1_intensity=min_ms1_intensity, model=model, print_reward=True, print_eval=True)
    print()

method = PPO max_peaks = 200 rt_tol = 120


Episode 0 (43060 chemicals)
steps	 500 	total rewards	 75.81979129173445
steps	 1000 	total rewards	 204.27003227361593
steps	 1500 	total rewards	 338.7380562625648
steps	 2000 	total rewards	 485.1248794975753
steps	 2500 	total rewards	 651.0979707992692
steps	 3000 	total rewards	 791.5385749063698
steps	 3500 	total rewards	 873.2517879853135
steps	 4000 	total rewards	 936.9050995760831
steps	 4500 	total rewards	 982.2431205635565
steps	 5000 	total rewards	 1023.1185589073896
Finished after 5229 timesteps with total reward 1047.134132338187
{'coverage_prop': '0.058', 'intensity_prop': '0.037', 'ms1/ms2 ratio': '0.605', 'efficiency': '0.772', 'TP': '1105', 'FP': '393', 'FN': '41562', 'precision': '0.738', 'recall': '0.026', 'f1': '0.050'}

method = topN max_peaks = 200 rt_tol = 120


Episode 0 (43060 chemicals)
steps	 500 	total rewards	 27.30326929199802
steps	 1000 	total rewards	 138.6030665041288
steps	 1500 	total rewards	 295.643

#### Test classic Top-N controller in ViMMS

In [20]:
mass_spec = IndependentMassSpectrometer(ionisation_mode, filtered)

In [ ]:
controller = TopNController(ionisation_mode, N, isolation_window, mz_tol, rt_tol, min_ms1_intensity)
env = Environment(mass_spec, controller, min_rt, max_rt, progress_bar=True, out_dir=out_dir, 
                  out_file='QCB_TopN_controller.mzML')
env.run()

  0%|          | 0/1440 [00:00<?, ?it/s]

#### Evaluation from mzML

Evaluation against the list of peaks picked from the fullscan QCB files.

There are two sets of parameters used for the peak picking: 'before' and 'after'.
'After' is more strict than 'before', with higher thresholds on intensity etc.

In [ ]:
fullscan_path = os.path.abspath('../fullscan_QCB.mzML')
fullscan_path

### 'Before' results

In [ ]:
aligned_file = os.path.abspath('../fullscan_QCB_box_before.csv')
aligned_file

In [ ]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_PPO_0.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

In [ ]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_TopN_0.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

In [ ]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_TopN_controller.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

In [ ]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_random_0.mzML'),  
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

### 'After' results

In [ ]:
aligned_file = os.path.abspath('../fullscan_QCB_box_after.csv')
aligned_file

In [ ]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_PPO_0.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

In [ ]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_TopN_0.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

In [ ]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_TopN_controller.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

In [ ]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_random_0.mzML'),  
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']